In [5]:
# Chipathon 2026 - NYCMOS
# This notebook computes the sizing for Gilbert cell mixer switching pair & LO quad based on gm/Id method
# Resource: https://github.com/bmurmann/Chipathon2025/tree/main

from pygmid import Lookup as lk
from scipy.io import loadmat
import numpy as np
import pandas as pd
import os

In [6]:
# Change file depending on where you cloned repo
n = lk("../lookup_tables/nfet_03v3.mat") #located in NYCMOS/lookup_tables/nfet_03v3.mat, replaced NYCMOS with .. because idk how to get paths to work in jupyter
p = lk('../lookup_tables/pfet_03v3.mat')
print(n._Lookup__DATA.keys())
print(n._Lookup__DATA.keys())

dict_keys(['INFO', 'CORNER', 'TEMP', 'VGS', 'VDS', 'VSB', 'L', 'W', 'NFING', 'ID', 'VT', 'GM', 'GMB', 'GDS', 'CGG', 'CGB', 'CGD', 'CGS', 'CDD', 'CSS', 'STH', 'SFL'])
dict_keys(['INFO', 'CORNER', 'TEMP', 'VGS', 'VDS', 'VSB', 'L', 'W', 'NFING', 'ID', 'VT', 'GM', 'GMB', 'GDS', 'CGG', 'CGB', 'CGD', 'CGS', 'CDD', 'CSS', 'STH', 'SFL'])


In [7]:
# Target for output amplifier

In [10]:
# Target
#assume 2.5k resistor load and ibias around 500 uA for ~1.25 V dc output
Rout = 2.5e3
gain = 5
gm_required = gain/Rout*np.array([1,1,1])
# gm_required = np.array([10e-3, 10e-3, 10e-3])
gm_id = np.array([5, 10, 15]) #weak to moderate inversion
l = 0.28
print(gm_required)
print(10e-3)

[0.002 0.002 0.002]
0.01


In [11]:
# Sizing for output transistor

id_required = gm_required / gm_id #required Id for the gm target and gm_id considered
Jd = p.lookup('ID_W', GM_ID=gm_id, L = l) # A/um current density for gm_id considered

print(Jd)

W = id_required / Jd # width required to match id_required
cgg_w = p.lookup('CGG_W', GM_ID=gm_id, L = l) # cgg/w for the given width

cgg = cgg_w * W # convert to absolute capacitance
ft = gm_required / (2 * np.pi * cgg) # Estimate transit freq

vdsat_est = 2/gm_id
print(gm_required.shape)

df = pd.DataFrame(
    [gm_required, gm_id, id_required*1e3, Jd, W, cgg, ft, vdsat_est],
    index=['gm_required', 'gm/id', 'id_required (mA)', 'Jd', 'W (um)', 'Cgg', 'ft', 'Vdsat'],
    columns = ['1','2','3']
)
df

[9.82439751e-06 2.58500770e-06 8.12233743e-07]
(3,)


,1,2,3
gm_required,2.000000e-03,2.000000e-03,2.000000e-03
gm/id,5.000000e+00,1.000000e+01,1.500000e+01
id_required (mA),4.000000e-01,2.000000e-01,1.333333e-01
Jd,9.824398e-06,2.585008e-06,8.122337e-07
W (um),4.071496e+01,7.736921e+01,1.641564e+02
Cgg,5.076786e-14,9.397659e-14,1.928405e-13
ft,6.269910e+09,3.387119e+09,1.650638e+09
Vdsat,4.000000e-01,2.000000e-01,1.333333e-01
